In [1]:

!pip install openai-whisper spacy transformers torch torchaudio -q
!python -m spacy download en_core_web_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 60.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
# These are public domain 911 call audio samples for testing
import os

# We'll generate 3 short synthetic-style test audio clips using gTTS
# (simulates what real 911 call transcription would produce)
!pip install gtts pydub -q

from gtts import gTTS

calls = [
    ("C001", "There is a fire, people are trapped on the second floor of the building on downtown avenue. Please hurry!"),
    ("C002", "I just witnessed a car accident on Main Street near the intersection of Oak Road. Two vehicles, someone is injured."),
    ("C003", "There is a robbery happening right now at the convenience store on Elm Street. The suspect has a weapon."),
]

os.makedirs("audio_files", exist_ok=True)
for call_id, text in calls:
    tts = gTTS(text=text, lang='en')
    tts.save(f"audio_files/{call_id}.mp3")
    print(f"Saved {call_id}.mp3")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.
Saved C001.mp3
Saved C002.mp3
Saved C003.mp3


In [3]:
import whisper

model = whisper.load_model("base")  # free, runs on CPU, ~140MB

transcripts = {}
for call_id, _ in calls:
    result = model.transcribe(f"audio_files/{call_id}.mp3")
    transcripts[call_id] = result["text"]
    print(f"{call_id}: {result['text']}")

/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


C001:  There is a fire. People are trapped on the second floor of the building on downtown Avenue. Please hurry.


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


C002:  I just witnessed a car accident on Main Street near the intersection of Oak Road. Two vehicles. Someone is injured.


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


C003:  There is a robbery happening right now at the convenience store on Elm Street. The suspect has a weapon.


In [4]:
!pip install --upgrade click
import spacy

nlp = spacy.load("en_core_web_sm")

def extract_info(text):
    doc = nlp(text)
    locations = [ent.text for ent in doc.ents if ent.label_ in ("GPE", "LOC", "FAC")]
    events = []

    # Simple keyword matching for incident type
    text_lower = text.lower()
    if any(w in text_lower for w in ["fire", "flame", "burning", "smoke"]):
        events.append("Fire")
    if any(w in text_lower for w in ["accident", "crash", "collision", "vehicle"]):
        events.append("Road Accident")
    if any(w in text_lower for w in ["robbery", "theft", "steal", "weapon", "gun"]):
        events.append("Robbery/Theft")
    if any(w in text_lower for w in ["trapped", "injured", "hurt", "medical"]):
        events.append("Medical Emergency")

    return {
        "location": ", ".join(locations) if locations else "Unknown",
        "event": ", ".join(events) if events else "Unknown Incident"
    }

extracted = {}
for call_id, transcript in transcripts.items():
    extracted[call_id] = extract_info(transcript)
    print(f"{call_id} → Event: {extracted[call_id]['event']} | Location: {extracted[call_id]['location']}")

  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
Using cached click-8.3.1-py3-none-any.whl (108 kB)
  Attempting uninstall: click
    Found existing installation: click 8.1.8
    Uninstalling click-8.1.8:
      Successfully uninstalled click-8.1.8
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gtts 2.5.4 requires click<8.2,>=7.1, but you have click 8.3.1 which is incompatible.
C001 → Event: Fire, Medical Emergency | Location: Avenue
C002 → Event: Road Accident, Medical Emergency | Location: Main Street, Oak Road
C003 → Event: Robbery/Theft | Location: Elm Street


In [5]:
from transformers import pipeline

sentiment_pipeline = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

# Urgency keyword scoring
urgency_keywords = ["help", "hurry", "please", "now", "trapped", "dying", "emergency",
                    "weapon", "fire", "injured", "robbery", "immediately"]

def get_urgency_score(text):
    text_lower = text.lower()
    hits = sum(1 for word in urgency_keywords if word in text_lower)
    return round(min(hits / 5, 1.0), 2)  # normalise to 0–1

results = {}
for call_id, transcript in transcripts.items():
    sentiment = sentiment_pipeline(transcript[:512])[0]  # distilbert max 512 tokens
    label = "Distressed" if sentiment["label"] == "NEGATIVE" else "Calm"
    urgency = get_urgency_score(transcript)
    results[call_id] = {
        "sentiment": label,
        "urgency_score": urgency
    }
    print(f"{call_id} → Sentiment: {label} | Urgency: {urgency}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

C001 → Sentiment: Distressed | Urgency: 0.8
C002 → Sentiment: Distressed | Urgency: 0.2
C003 → Sentiment: Distressed | Urgency: 0.6


In [6]:
import pandas as pd

rows = []
for call_id, transcript in transcripts.items():
    rows.append({
        "Call_ID": call_id,
        "Transcript": transcript.strip(),
        "Extracted_Event": extracted[call_id]["event"],
        "Location": extracted[call_id]["location"],
        "Sentiment": results[call_id]["sentiment"],
        "Urgency_Score": results[call_id]["urgency_score"]
    })

df = pd.DataFrame(rows)
df.to_csv("audio_output.csv", index=False)
print("Saved audio_output.csv")
df

Saved audio_output.csv


,Call_ID,Transcript,Extracted_Event,Location,Sentiment,Urgency_Score
0,C001,There is a fire. People are trapped on the sec...,"Fire, Medical Emergency",Avenue,Distressed,0.8
1,C002,I just witnessed a car accident on Main Street...,"Road Accident, Medical Emergency","Main Street, Oak Road",Distressed,0.2
2,C003,There is a robbery happening right now at the ...,Robbery/Theft,Elm Street,Distressed,0.6


In [7]:
from google.colab import files

files.download("audio_output.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>